# 第一课｜从膜电位到一个最小计算神经元

欢迎来到 FPGA FlyBrain 的第一课。

这门课的出发点不是计算机，而是你已经见过的神经生理学：膜电位会变化，突触输入会影响神经元，达到一定条件后会产生动作电位。我们要做的事情，是一步一步把这些熟悉的概念翻译成**数学模型 → 程序 → 数字电路 → 最终运行在真实硬件上的神经网络**。

今天先不学硬件。今天只回答一个问题：

> **如果我们的目标是研究大规模神经网络的计算，最少需要保留神经元的哪些性质？**

本课的主要新概念只有一个：**科学模型是一种有目的的简化。**

## 1. 你现在看到的是什么：Jupyter Notebook

你正在阅读的是一个 **Jupyter Notebook**。它可以把两种东西放在同一个页面里：

- **Markdown 单元格**：像教材一样写解释、公式、问题和结论；
- **Code 单元格**：可以真正运行 Python 代码并马上看到结果。

所以这不是“先读一本书，再去另一个地方写代码”。这里本身就是一本**可以运行的教材**。

如果你以前没有用过 Jupyter，没有关系。现阶段只要知道：选中一个代码单元格，运行它，就会得到输出。Python 语法也不要求提前学完，我们会在需要时解释。

## 2. 我们最终要去哪里：什么是 FPGA？

先认识终点，但今天不用掌握它。

**现场可编程门阵列（Field-Programmable Gate Array, FPGA）** 是一种可以在制造完成以后，根据我们的设计重新配置内部数字电路的芯片。

普通电脑上的 **中央处理器（Central Processing Unit, CPU）** 通常按照一条条指令执行程序；FPGA 的特别之处在于，我们可以把某些计算直接组织成自己的并行数字电路。

这个项目最终想做的是：把神经元状态、突触事件和网络传播真正变成 FPGA 里的硬件结构。

**但现在先把这句话放在脑后。** 第一课只需要理解神经元模型。我们不会因为终点是 FPGA，就在第一天塞给你数字电路术语。

## 3. 从你已经知道的神经生理开始

你大概已经知道这些事情：

1. 神经元膜两侧存在电位差，我们常把它描述成**膜电位**；
2. 突触输入可以让膜电位朝更容易或更不容易放电的方向变化；
3. 当膜电位达到某种条件时，神经元可以产生动作电位；
4. 动作电位之后，神经元状态会发生变化，并存在一定的不应期行为。

真实神经元当然远比这复杂：离子通道、树突、不同神经递质、适应、突触动力学、细胞类型差异……都很重要。

但工程问题是：

> 如果我们一开始就把所有生物细节全部保留，还能不能看清楚“网络连接怎样产生计算”？

这就是为什么我们需要模型。

## 4. 什么是模型？

一个**模型（model）**不是在宣称“真实世界只有这些东西”。模型是在说：

> 为了回答当前问题，我们暂时保留某些因素，忽略另一些因素。

地图就是一个很好的例子。地铁图不画每一栋楼，但它仍然非常有用，因为它保留了“哪一站和哪一站相连”这个当前最重要的信息。

我们的第一版神经元模型也一样：它不是为了重建完整细胞生物物理，而是为了得到一个足够简单、又能产生状态变化和 spike 事件的计算单元。

## 5. 什么是 LIF？把名字拆开看

我们先使用一种经典的简化神经元模型：

**漏电积分发放模型（Leaky Integrate-and-Fire, LIF）**。

这个名字其实已经把模型的三个核心动作说完了：

- **Leaky，漏电/衰减**：如果没有足够的新输入，过去积累的膜电位影响会逐渐减弱；
- **Integrate，积分/累积**：新的输入会被累积到当前状态上；
- **Fire，发放**：当状态达到阈值时，模型产生一次 spike 事件。

这里的 **spike** 可以理解成“神经元在这个时刻发生了一次放电事件”。注意，我们并没有模拟真实动作电位那条快速上升再下降的完整电压波形；我们只保留“发生了一次事件”这件事。

这正是模型的第一处重要简化。

## 6. 用一个离散时间公式表达它

我们先把时间切成一小步一小步，用 `t = 0, 1, 2, ...` 表示。这样的模型叫**离散时间（discrete time）**模型。

在每一步，我们先计算：

`V[t+1] = alpha × V[t] + I[t]`

每个符号都是什么意思？

- `V[t]`：第 `t` 步开始时模型保存的膜电位状态；
- `I[t]`：这一时间步收到的输入；
- `alpha`：上一时刻状态保留下来的比例。`alpha < 1` 时，旧状态会逐渐衰减；
- `V[t+1]`：加入衰减和新输入后得到的下一状态候选值。

然后我们问：

`V[t+1] >= threshold ?`

如果达到阈值，就产生一次 spike，并把膜电位按照 reset 规则复位。

第一课暂时不加入 refractory period（不应期）的额外状态，因为我们希望先把最小模型看清楚。后面会专门讨论“哪些规则必须在实现前说清楚”。

## 7. 把公式翻译成最小 Python 程序

下面的代码只做三件事：

1. 保存当前状态 `v`；
2. 每一步计算 `candidate_v = alpha * v + current`；
3. 判断是否达到 threshold，如果达到就记录 spike 并 reset。

先不用担心 Python 的每个语法细节。先看它和上面的模型是否一一对应。

In [ ]:
def run_lif(inputs, alpha=0.9, threshold=1.0, reset=0.0):
    v = 0.0
    trace = []

    for t, current in enumerate(inputs):
        candidate_v = alpha * v + current
        spike = candidate_v >= threshold
        stored_v = reset if spike else candidate_v

        trace.append({
            't': t,
            'input': current,
            'candidate_v': candidate_v,
            'stored_v': stored_v,
            'spike': spike,
        })

        v = stored_v

    return trace

inputs = [0.22] * 15
trace = run_lif(inputs)

for row in trace:
    print(row)

## 8. 先读结果，不急着画图

看每一行输出时，重点找五件事：

- `t`：现在是第几步；
- `input`：这一时刻输入多少；
- `candidate_v`：threshold 判断之前的膜电位候选值；
- `spike`：这一时刻是否发放；
- `stored_v`：这个时间步结束后真正保存、交给下一步的状态。

特别注意 `candidate_v` 和 `stored_v` 不一定相同：如果 spike 发生，模型会把状态 reset。

这以后会变成一个非常重要的硬件概念：**计算出来的 next state 和最后保存的 state 可以是两件不同的事。** 现在只需要先看到这个现象。

## 9. 把膜电位轨迹画出来

数字表很精确，但图更容易建立直觉。下面使用 Python 常见的绘图库 `matplotlib`。这里不要求你学习绘图库本身，只用它观察模型。

In [ ]:
import matplotlib.pyplot as plt

times = [row['t'] for row in trace]
candidate_v = [row['candidate_v'] for row in trace]
spike_times = [row['t'] for row in trace if row['spike']]
spike_values = [row['candidate_v'] for row in trace if row['spike']]

plt.figure(figsize=(9, 4))
plt.plot(times, candidate_v, marker='o', label='membrane candidate V')
plt.axhline(1.0, linestyle='--', label='threshold')
plt.scatter(spike_times, spike_values, marker='x', s=80, label='spike event')
plt.xlabel('time step')
plt.ylabel('model membrane state')
plt.title('A minimal LIF neuron')
plt.legend()
plt.show()

## 10. Observe：你应该看到什么？

运行以后，不要只看“图画出来了”。请回答：

1. 持续相同输入时，膜电位为什么不是简单地每次加 `0.22`？
2. `alpha = 0.9` 在模型里造成了什么效果？
3. spike 为什么只记录为一个时刻，而不是一条完整动作电位波形？
4. spike 后为什么下一轮又从较低状态开始？

如果这四个问题不能回答，先不要继续改代码。

## 11. Try It：先预测，再运行

请选择一次只改一个参数：

- 把 `alpha` 从 `0.9` 改成 `0.5`；
- 或把 `threshold` 从 `1.0` 改成 `1.5`；
- 或把每一步输入从 `0.22` 改成 `0.35`。

**先写下你的预测：spike 会更早、更晚，还是完全不出现？为什么？**

然后再运行。

这一步很重要：工程学习不是“改参数看看会发生什么”，而是“先根据模型预测，再用实验检验自己的理解”。

## 12. AI Task：AI 可以帮什么？

你可以让 AI：

- 把表格输出改得更漂亮；
- 增加不同 `alpha` 的对比图；
- 增加 refractory period 的实验版本；
- 草拟针对 no-input decay、threshold crossing、reset 的测试。

但先给 AI 一个约束：

> 不要擅自改变当前 LIF 的 update 顺序、threshold 规则或 reset 规则；如果认为需要改变，先解释为什么，并把它当成一个待讨论的 specification change。

AI 可以帮我们写，但不能偷偷替我们定义模型。

## 13. Human Check：不用 AI，你应该能回答

- LIF 的三个词 Leaky / Integrate / Fire 分别对应什么？
- `V` 是真实细胞膜电位的完整复制吗？为什么不是？
- `alpha` 在这个简化模型里表达了什么？
- spike 在这里为什么是一个 event，而不是动作电位波形？
- threshold 和 reset 是自然定律自动告诉程序的吗，还是我们必须明确规定的模型规则？
- 这个模型目前故意忽略了哪些你在生理学中学过的东西？

## 14. Engineering Handoff

Notebook 里的代码现在还是教学原型。等我们确认模型语义以后，正式实现会进入：

`python/reference/lif_float.py`

之后 Notebook 应该 import 正式模块，而不是长期维护另一份稍微不同的实现。

## 15. 项目追踪 Project Trace

这一段是给项目维护者和未来的你看的，不是本课需要背的内容。

- Lesson ID: `LSN-001`
- Engineering slice: `RMD-001`
- Product requirement/design: `FR1 / DP1`
- Initial tests: `T-001 ~ T-004`

这些 ID 用来让教学、需求、代码和测试长期保持对应。

## 16. Exit Ticket

进入下一课之前，你应该能够：

1. 用自己的话解释什么是科学模型；
2. 展开并解释 **Leaky Integrate-and-Fire (LIF)**；
3. 说出公式中 `V`、`I`、`alpha`、threshold、reset 分别代表什么；
4. 给定很短的一组输入，手工推演几步膜电位和 spike；
5. 说明这个模型保留了什么、故意丢掉了什么。

如果这些都可以，我们下一课再问一个新问题：

> 这些 `0.9`、`0.22`、`1.0` 到了真实数字硬件里，到底怎样保存？